# 🚀 Data Engineering Pipeline Tutorial - Part 1
## Configuration & Data Extraction

This tutorial uses **real working code** from our pipeline that successfully:
- Extracted 23 rows from MySQL, CSV, and API sources
- Uploaded to S3 bucket `tina-data-lake-381324498760`
- Loaded to PostgreSQL warehouse

### Your Infrastructure:
```
┌─────────────────────────────────────────────────────────┐
│  MySQL (mysql:3306)     → customers, orders tables  │
│  PostgreSQL (postgres:5432) → warehouse destination    │
│  AWS S3 (ap-southeast-2)    → tina-data-lake-381324498760│
└─────────────────────────────────────────────────────────┘
```

---
## 📦 Part 1: Configuration Management

**Golden Rule:** Never hardcode credentials. Use environment variables.

**Why?**
- Security: Credentials don't end up in git
- Flexibility: Different values for dev/staging/prod
- Best practice: 12-factor app methodology

In [ ]:
# config.py - Your actual working configuration
import os

# os.getenv("VAR_NAME", "default_value")
# - First tries to read from environment variable
# - Falls back to default if not set

# AWS Settings - These work with your AWS account
AWS_REGION = os.getenv("AWS_REGION", "ap-southeast-2")
S3_BUCKET = os.getenv("S3_BUCKET", "tina-data-lake-381324498760")  # Your actual bucket!

# Medallion Architecture Zones
BRONZE_ZONE = "bronze"   # Raw data exactly as received
SILVER_ZONE = "silver"   # Cleaned and validated
GOLD_ZONE = "gold"       # Business-ready, transformed

# Your PostgreSQL warehouse (running in Docker)
POSTGRES_CONFIG = {
    "host": os.getenv("PG_HOST", "localhost"),
    "port": int(os.getenv("PG_PORT", 5432)),
    "database": os.getenv("PG_DATABASE", "devdb"),
    "user": os.getenv("PG_USER", "devuser"),
    "password": os.getenv("PG_PASSWORD", "devpassword")
}

# Your MySQL source database (running in Docker)
MYSQL_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "localhost"),
    "port": int(os.getenv("MYSQL_PORT", 3306)),
    "database": os.getenv("MYSQL_DATABASE", "devdb"),
    "user": os.getenv("MYSQL_USER", "devuser"),
    "password": os.getenv("MYSQL_PASSWORD", "devpassword")
}

print(f"✅ Config loaded")
print(f"   S3 Bucket: {S3_BUCKET}")
print(f"   Region: {AWS_REGION}")

### 💡 Why Dictionary for DB Config?

```python
# Dictionary allows **kwargs unpacking:
conn = psycopg2.connect(**POSTGRES_CONFIG)

# Is equivalent to:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="devdb",
    user="devuser",
    password="devpassword"
)
```

Much cleaner and easier to manage!

---
## 🔧 Part 2: The Retry Decorator Pattern

**Real Problem:** Our pipeline makes network calls that can fail:
- Database connections timeout
- APIs rate-limit or have temporary outages
- S3 uploads occasionally fail

**Solution:** Retry with exponential backoff

In [ ]:
import time

def retry_on_failure(max_retries: int = 3, delay: int = 5):
    """
    Decorator that retries a function on failure.
    
    DECORATOR = A function that wraps another function to add behavior.
    
    Exponential backoff pattern:
    - Attempt 1 fails → wait 5 seconds
    - Attempt 2 fails → wait 10 seconds (5 * 2^1)
    - Attempt 3 fails → wait 20 seconds (5 * 2^2)
    - Give up after max_retries
    
    Why exponential? Gives the failing service time to recover.
    """
    def decorator(func):          # Takes the function to wrap
        def wrapper(*args, **kwargs):  # Replacement function
            last_exception = None
            
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)  # Try original function
                except Exception as e:
                    last_exception = e
                    wait_time = delay * (2 ** attempt)  # Exponential: 5, 10, 20...
                    print(f"⚠️ Attempt {attempt + 1} failed: {e}")
                    print(f"   Retrying in {wait_time}s...")
                    time.sleep(wait_time)
            
            raise last_exception  # All retries exhausted
        return wrapper
    return decorator

# USAGE EXAMPLE:
@retry_on_failure(max_retries=3, delay=1)
def flaky_api_call():
    """Simulates an unreliable API."""
    import random
    if random.random() < 0.7:  # 70% failure rate
        raise ConnectionError("API timeout!")
    return {"status": "success", "data": [1, 2, 3]}

# Try it (may take a few attempts)
try:
    result = flaky_api_call()
    print(f"✅ Success: {result}")
except Exception as e:
    print(f"❌ All retries failed: {e}")

### 🎯 Decorator Syntax Explained

```python
# This:
@retry_on_failure(max_retries=3)
def my_function():
    pass

# Is exactly the same as:
def my_function():
    pass
my_function = retry_on_failure(max_retries=3)(my_function)
```

**When to use in Data Engineering:**
- Database connections
- API calls (we used it for exchange rate API)
- S3 uploads/downloads
- Any network operation

---
## 📊 Part 3: Database Extraction

### 3.1 Connecting to Your MySQL Database

This code extracted **5 customers** and **7 orders** from your MySQL.

In [ ]:
import mysql.connector
import pandas as pd

# Your actual MySQL config
MYSQL_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "database": "devdb",
    "user": "devuser",
    "password": "devpassword"
}

# Connect using **kwargs unpacking
conn = mysql.connector.connect(**MYSQL_CONFIG)
print("✅ Connected to MySQL")

# Check what tables exist
cursor = conn.cursor()
cursor.execute("SHOW TABLES")
tables = [t[0] for t in cursor.fetchall()]
print(f"📋 Available tables: {tables}")

conn.close()

### 3.2 Memory-Efficient Extraction with Generators

**Problem:** Tables can have millions of rows. Loading all at once crashes memory.

**Solution:** Use **generators** to process in batches.

```python
# Regular function - loads ALL data into memory
def get_all_data():
    return [row for row in million_rows]  # 💥 Memory crash!

# Generator - processes one batch at a time
def get_data_batches():
    for batch in chunks:
        yield batch  # ✅ Only one batch in memory
```

In [ ]:
from typing import Generator
from datetime import datetime

class DatabaseExtractor:
    """
    Extract data from databases in memory-efficient batches.
    
    This is the actual class used in our pipeline!
    """
    
    def __init__(self, db_type: str = "mysql"):
        self.db_type = db_type
        self.config = MYSQL_CONFIG
        self.conn = None
    
    def connect(self):
        """Establish database connection."""
        self.conn = mysql.connector.connect(**self.config)
        print(f"✅ Connected to {self.db_type}")
    
    def close(self):
        """Always close connections when done!"""
        if self.conn:
            self.conn.close()
            print(f"✅ Closed {self.db_type} connection")
    
    def extract_table(self, table: str, batch_size: int = 10000) -> Generator[pd.DataFrame, None, None]:
        """
        Extract table data in batches using a GENERATOR.
        
        The 'yield' keyword makes this a generator:
        - Function pauses at yield, returns value
        - Resumes from where it left off on next call
        - Only one batch in memory at a time!
        """
        if not self.conn:
            self.connect()
        
        # Get total count
        cursor = self.conn.cursor()
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        total_rows = cursor.fetchone()[0]
        cursor.close()
        
        print(f"📊 Extracting {total_rows} rows from {table}")
        
        offset = 0
        while offset < total_rows:
            # LIMIT/OFFSET for pagination
            query = f"SELECT * FROM {table} LIMIT {batch_size} OFFSET {offset}"
            df = pd.read_sql(query, self.conn)
            
            if df.empty:
                break
            
            # Add metadata - ALWAYS track data lineage!
            df['_extracted_at'] = datetime.now()
            df['_source_table'] = table
            df['_source_db'] = self.db_type
            
            yield df  # YIELD instead of RETURN
            
            offset += batch_size
            print(f"   Extracted {min(offset, total_rows)}/{total_rows} rows")

# ACTUAL USAGE from our pipeline run:
extractor = DatabaseExtractor("mysql")
extractor.connect()

# Process customers table
for batch_df in extractor.extract_table("customers"):
    print(f"\n📦 Batch received: {len(batch_df)} rows")
    print(batch_df.head(2))  # Show first 2 rows

extractor.close()

### 3.3 API Extraction

Our pipeline extracted exchange rates from a real API:

In [ ]:
import requests
import pandas as pd
from datetime import datetime

class APIExtractor:
    """Extract data from REST APIs."""
    
    @staticmethod  # No 'self' needed - pure function
    def extract(url: str, headers: dict = None) -> pd.DataFrame:
        """
        Fetch JSON from API and convert to DataFrame.
        
        Type hints (url: str, -> pd.DataFrame):
        - Document expected types
        - Enable IDE autocomplete
        - Catch errors early
        """
        print(f"🌐 Fetching: {url}")
        
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()  # Raises exception if HTTP error
        
        data = response.json()
        
        # Handle different JSON structures
        if isinstance(data, list):
            df = pd.DataFrame(data)
        elif isinstance(data, dict):
            # pd.json_normalize flattens nested JSON
            df = pd.json_normalize(data)
        
        # Add metadata
        df['_extracted_at'] = datetime.now()
        df['_source_api'] = url
        
        print(f"✅ Extracted {len(df)} records")
        return df

# ACTUAL API call from our pipeline:
df = APIExtractor.extract("https://api.exchangerate-api.com/v4/latest/USD")
print(f"\nColumns: {list(df.columns)[:5]}...")  # Show first 5 columns
print(f"Base currency: {df['base'].values[0]}")

### 3.4 CSV Extraction

In [ ]:
class CSVExtractor:
    """Extract data from CSV files."""
    
    @staticmethod
    def extract(file_path: str, **kwargs) -> pd.DataFrame:
        """
        Read CSV file into DataFrame.
        
        **kwargs passes any extra arguments to pd.read_csv:
        - delimiter=';'
        - encoding='utf-8'
        - parse_dates=['date_column']
        """
        print(f"📄 Reading: {file_path}")
        df = pd.read_csv(file_path, **kwargs)
        df['_extracted_at'] = datetime.now()
        df['_source_file'] = file_path
        print(f"✅ Extracted {len(df)} rows")
        return df

# Read our sample CSV files
import os
base_dir = os.path.dirname(os.path.abspath('__file__'))

customers_df = CSVExtractor.extract("data/sources/customers.csv")
print(customers_df)

products_df = CSVExtractor.extract("data/sources/products.csv")
print(products_df)

---
## 🎯 Part 1 Summary

| Concept | Syntax | Real Usage |
|---------|--------|------------|
| Environment vars | `os.getenv("VAR", "default")` | DB credentials, S3 bucket |
| Dict unpacking | `func(**config)` | Database connections |
| Decorator | `@retry_on_failure()` | API calls, DB queries |
| Generator | `yield batch` | Memory-efficient extraction |
| Type hints | `def func(x: str) -> pd.DataFrame` | Documentation, IDE support |
| @staticmethod | `@staticmethod` | Utility functions in classes |

### ✅ What We Extracted (Actual Results):
- MySQL: 5 customers + 7 orders = 12 rows
- CSV: 5 customers + 5 products = 10 rows  
- API: 1 exchange rate record
- **Total: 23 rows** → Bronze zone in S3